In [ ]:
from hyper_evolution import time_evolve, animate
from scipy.special import gamma
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
lam = 3
x0, p0 = 0., np.sqrt(lam-0.25)
breadth = 1000

In [ ]:
# domain = np.linspace(-breadth, breadth, 10000)
# dx = domain[1] - domain[0]
dx = .05
domain = np.arange(-breadth, breadth + dx, dx)

def sech2(x):
    ax = np.abs(x)
    e = np.exp(-ax)
    return (2 * e / (1 + e*e))**2

Vx = -lam * (lam + 1) * sech2(domain) + lam**2

c = np.sqrt(gamma(lam+0.5)/(np.sqrt(np.pi) * gamma(lam)))

y = domain - x0
ay = np.abs(y)
log_cosh = ay + np.log1p(np.exp(-2*ay)) - np.log(2)

psi_0 = np.exp(np.log(c) - lam * log_cosh)
psi_0 = psi_0.astype(np.complex128)
psi_0 *= np.exp(1j * p0 * domain)


In [ ]:
max_t = 10
dt = dx ** 2
T = int(max_t / dt) + 1
results = time_evolve(domain, psi_0, Vx, max_t, it=T, hyperbolic=True, lam_val=lam)

In [ ]:
t_eval = np.array(results['t'])
E_q = np.array(results['T']) + np.array(results['V'])
unbound_prob = np.abs(results['unbound_psi']) ** 2
unbound_x = np.array(results['unbound_x'])
left_diff = (unbound_prob[1:, 0] - unbound_prob[:-1, 0])
right_diff = (unbound_prob[1:, -1] - unbound_prob[:-1, -1])

In [ ]:
plt.plot(t_eval, results['err_psi'], label='err_psi')
plt.plot(t_eval, results['err_norm'], label='err_norm')
plt.legend()
plt.show()

In [ ]:
unbound_contribution = [np.trapz(abs(results['unbound_psi'][i])**2, dx=dx) for i in range(len(t_eval))]
plt.plot(t_eval, unbound_contribution)

In [ ]:
unbound = results['unbound_x']
# hit_boundary = np.where(abs(left_diff) + abs(right_diff) > 0)
# if len(hit_boundary[0]) > 0:
#     hit_boundary = hit_boundary[0][0]
# else:
hit_boundary = len(unbound)

plt.plot(t_eval[:hit_boundary], unbound[:hit_boundary])
c = np.polyfit(t_eval[:hit_boundary], unbound[:hit_boundary], 1)
poly = np.poly1d(c)
plt.plot(t_eval[:hit_boundary], poly(t_eval[:hit_boundary]), linestyle='--')
plt.title('Expectation of unbound x')
plt.show()

In [ ]:
c

data = {'p0': [], 'unbound_prob': [], 'm': [], 'b': [], 'lambda': [], 'unbound_p': []}

In [ ]:
data['p0'] += [p0]
data['unbound_prob'] += [unbound_contribution[0]]
data['m'] += [c[0]]
data['b'] += [c[1]]
data['lambda'] += [lam]
data['unbound_p'] += [np.array(results['unbound_p'])]
with open(f'results.npy', 'wb') as f:
    np.save(f, data)

In [ ]:
stop here

In [ ]:
p0 = 1

In [ ]:
from scipy.integrate import solve_ivp

def hamilton_eqs(_, y):
    x, p = y
    dxdt = 2 * p
    dpdt = -2 * lam**2 * np.tanh(x) * (1 - np.tanh(x)**2) 
    return [dxdt, dpdt]

sol = solve_ivp(
    hamilton_eqs,
    t_span=(0, max_t),
    y0=[x0, p0],
    t_eval=t_eval,
    method="DOP853",
    rtol=1e-10,
    atol=1e-12
)

x_classical = sol.y[0]
p_classical = sol.y[1]

In [ ]:
p_classical[-1] - p_classical[0]

In [ ]:
plt.plot(p_classical)

In [ ]:
plt.plot(t_eval, np.gradient(results['unbound_x'], dt))
plt.plot(t_eval, 2 * np.array(results['unbound_p']))

In [ ]:
plt.figure(figsize=(10, 6), dpi=100)
plt.plot(t_eval, x_classical, '--', c='g', label='Classical')
plt.plot(t_eval, results['x'], label='Quantum')
plt.title(f'Hyper Correspondence with Matched Energies ($x_0\\approx$ {x0:.2f}, $p_0 \\approx$ {p0:.2f}, $\\lambda = {lam}$)')
plt.ylabel("Position ⟨x⟩")
plt.xlabel("Time")
plt.grid(True, alpha=0.5)
plt.legend()
plt.show()

In [ ]:
skip = 10
# animate(domain, unbound_prob[::skip], max_t, max_x=10, filename='unbound.gif')

In [ ]:
plt.plot(t_eval, unbound_edge_left, label='Left Edge % Change')
plt.plot(t_eval, unbound_edge_right, label='Right Edge % Change')
# plt.plot(t_eval, unbound_x*10, label='Unbound ⟨x⟩ (scaled)', alpha=0.7)
plt.title(f'Domain: {breadth}, $\Delta t$: {dt:.5f}, $\Delta x$: {dx:.2f}')
plt.xlabel('Time')
plt.ylabel('Percentage Change')
plt.grid(True, alpha=0.5)
plt.legend()
plt.show()

# np.save(f"results_{breadth}_{dt}.npy", results)